In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import albumentations as A

# --- 1. DATA PREPARATION AND AUGMENTATION ---
# Augmentation helps the model generalize by learning tumor patterns from different angles/perspectives
aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
    A.OneOf([
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
        A.GridDistortion(p=0.5),
    ], p=0.3)
])

base_path = '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m'
all_masks = glob.glob(os.path.join(base_path, "*/*_mask.tif"))
image_files = [f.replace('_mask.tif', '.tif') for f in all_masks]

df = pd.DataFrame({'image_path': image_files, 'mask_path': all_masks})

def check_tumor(mask_path):
    """Checks if the mask contains a tumor (max value > 0)"""
    try:
        mask = cv2.imread(mask_path, 0)
        return 1 if np.max(mask) > 0 else 0
    except: return -1

# Filtering and splitting the dataset
df['has_tumor'] = [check_tumor(m) for m in tqdm(df['mask_path'], desc="Analyzing Masks")]
df = df[df['has_tumor'] != -1].reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['has_tumor'])
train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=42, stratify=train_df['has_tumor'])

# --- 2. ATTENTION U-NET ARCHITECTURE ---
# Attention Gate: Suppresses irrelevant background regions and focuses on the tumor features
def attention_gate(x, gating, inter_shape):
    shape_x = tf.keras.backend.int_shape(x)
    shape_g = tf.keras.backend.int_shape(gating)

    phi_g = layers.Conv2D(inter_shape, (1, 1), padding='same')(gating)
    theta_x = layers.Conv2D(inter_shape, (3, 3), strides=(shape_x[1] // shape_g[1], shape_x[2] // shape_g[2]), padding='same')(x)

    add_xg = layers.add([phi_g, theta_x])
    act_xg = layers.Activation('relu')(add_xg)
    psi = layers.Conv2D(1, (1, 1), padding='same')(act_xg)
    sigmoid_xg = layers.Activation('sigmoid')(psi)
    
    upsample_psi = layers.UpSampling2D(size=(shape_x[1] // tf.keras.backend.int_shape(sigmoid_xg)[1], shape_x[2] // tf.keras.backend.int_shape(sigmoid_xg)[2]))(sigmoid_xg)
    y = layers.multiply([upsample_psi, x])
    return y

def conv_block(inputs, filters):
    """Standard Convolutional Block: Conv2D -> BatchNorm -> ReLU (twice)"""
    x = layers.Conv2D(filters, (3, 3), padding='same', kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def attention_unet(input_size=(256, 256, 3)):
    inputs = layers.Input(input_size)
    
    # Encoder Path (Downsampling)
    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    # Bottleneck
    b1 = conv_block(p3, 256)
    
    # Decoder Path with Attention (Upsampling)
    g1 = layers.UpSampling2D((2, 2))(b1)
    a1 = attention_gate(c3, g1, 128)
    u1 = layers.concatenate([g1, a1])
    c4 = conv_block(u1, 128)
    
    g2 = layers.UpSampling2D((2, 2))(c4)
    a2 = attention_gate(c2, g2, 64)
    u2 = layers.concatenate([g2, a2])
    c5 = conv_block(u2, 64)
    
    g3 = layers.UpSampling2D((2, 2))(c5)
    a3 = attention_gate(c1, g3, 32)
    u3 = layers.concatenate([g3, a3])
    c6 = conv_block(u3, 32)
    
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c6)
    return models.Model(inputs, outputs)

# --- 3. LOSS FUNCTION AND DATA GENERATOR ---
def dice_coef(y_true, y_pred):
    """Calculates the Dice Coefficient for segmentation accuracy"""
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + 1.0) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + 1.0)

def bce_dice_loss(y_true, y_pred):
    """Hybrid Loss: Combination of Binary Crossentropy and Dice Loss"""
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + (1 - dice_coef(y_true, y_pred))

def train_generator(df, batch_size=16, augment=False):
    """Custom Data Generator with optional real-time augmentation"""
    while True:
        df = df.sample(frac=1).reset_index(drop=True)
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i : i + batch_size]
            images, masks = [], []
            for _, row in batch_df.iterrows():
                img = cv2.imread(row['image_path'])
                img = cv2.resize(img, (256, 256))
                mask = cv2.imread(row['mask_path'], 0)
                mask = cv2.resize(mask, (256, 256))
                
                if augment:
                    augmented = aug(image=img, mask=mask)
                    img, mask = augmented['image'], augmented['mask']
                
                images.append(img / 255.0)
                masks.append(np.expand_dims(mask / 255.0, axis=-1))
            yield np.array(images), np.array(masks)

# --- 4. TRAINING CONFIGURATION ---
model = attention_unet()
model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss=bce_dice_loss, metrics=[dice_coef])

# Callbacks for robust training
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("best_attention_unet.h5", save_best_only=True, monitor='val_dice_coef', mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-7),
    tf.keras.callbacks.EarlyStopping(monitor='val_dice_coef', patience=10, restore_best_weights=True, mode='max')
]

history = model.fit(
    train_generator(train_df, 16, augment=True),
    steps_per_epoch=len(train_df) // 16,
    epochs=50,
    validation_data=train_generator(val_df, 16, augment=False),
    validation_steps=len(val_df) // 16,
    callbacks=callbacks
)

# --- 5. VISUALIZATION OF RESULTS ---

# A. Training Curves (Dice Coefficient and Loss)
def plot_training_history(history):
    plt.figure(figsize=(16, 6))
    
    # Plot Dice Coefficient
    plt.subplot(1, 2, 1)
    plt.plot(history.history['dice_coef'], label='Train Dice', color='#1f77b4', linewidth=2)
    plt.plot(history.history['val_dice_coef'], label='Val Dice', color='#ff7f0e', linewidth=2)
    plt.title('Training Process: Dice Coefficient', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Dice Score', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    plt.plot(history.history['val_loss'], label='Val Loss', color='#ff7f0e', linewidth=2)
    plt.title('Training Process: Loss', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss Value', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# B. Prediction Visualization on Random Test Samples
def visualize_predictions(model, df, n_samples=5):
    plt.figure(figsize=(20, n_samples * 5))
    
    # Select random tumorous samples for impact
    samples = df[df['has_tumor'] == 1].sample(n_samples)
    
    for i, (idx, row) in enumerate(samples.iterrows()):
        img = cv2.imread(row['image_path'])
        img_res = cv2.resize(img, (256, 256))
        img_input = np.expand_dims(img_res / 255.0, axis=0)
        
        true_mask = cv2.imread(row['mask_path'], 0)
        true_mask = cv2.resize(true_mask, (256, 256))
        
        pred = model.predict(img_input, verbose=0)
        pred_mask = (pred[0, :, :, 0] > 0.5).astype(np.uint8) # 0.5 Decision Threshold
        
        # 1. Original MRI
        plt.subplot(n_samples, 4, i*4 + 1)
        plt.imshow(img)
        plt.title(f"Sample {i+1}: Original MRI")
        plt.axis('off')
        
        # 2. Ground Truth
        plt.subplot(n_samples, 4, i*4 + 2)
        plt.imshow(true_mask, cmap='gray')
        plt.title("Ground Truth (Expert)")
        plt.axis('off')
        
        # 3. Model Prediction
        plt.subplot(n_samples, 4, i*4 + 3)
        plt.imshow(pred_mask, cmap='hot')
        plt.title("Attention U-Net Prediction")
        plt.axis('off')
        
        # 4. Result Overlay
        plt.subplot(n_samples, 4, i*4 + 4)
        plt.imshow(img)
        plt.imshow(pred_mask, cmap='Reds', alpha=0.4)
        plt.title("Overlay (MRI + Prediction)")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Run visualization
plot_training_history(history)
visualize_predictions(model, test_df, n_samples=3)